# Chapter 6 &mdash; Complementation: Swap $F$ and $Q-F$, but Totalize First

**Concept 1 of the Chapter 6 decomposition:** *Complementation of DFA: Swap Final and Non-Final, but Totalize First*

Flipping finality complements the language &mdash; but only on a totalized DFA, or black holes become accepting.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Complementation/Concept-Complementation.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


To complement a DFA, **swap final and non-final states**: $F' = Q - F$.

The catch is **totality**. If $\delta$ is partial, a string that "falls off" the
machine is rejected by *neither* set of final states, so the flip does not complement
anything. Worse, if you totalize *after* flipping, the black hole is created
**non-final** and stays non-final &mdash; and strings that should now be accepted are not.

So the order is fixed: **totalize, then flip.** Jove's `comp_dfa` does both.

## 2. Definitions

even0 = md2mc('''DFA
IF  : 0 -> Od
IF  : 1 -> IF
Od  : 0 -> IF
Od  : 1 -> Od
''')

A two-state machine, already total over $\{0,1\}$.

In [ ]:
even0 = md2mc('''DFA
IF  : 0 -> Od
IF  : 1 -> IF
Od  : 0 -> IF
Od  : 1 -> Od
''')
print("Sigma:", sorted(even0["Sigma"]), " F:", sorted(even0["F"]))

### A deliberately **partial** machine, to see the trap

In [ ]:
partial = md2mc('''DFA
IF : 0 -> Od
Od : 0 -> IF
''')
wide = addtosigma_dfa(partial, {'1'})
print("wide is total?", len(wide["Delta"]) == len(wide["Q"]) * len(wide["Sigma"]))

### Flip by hand, so the two orders can be compared

In [ ]:
def flip(D):
    E = dict(D); E["F"] = D["Q"] - D["F"]; return E

## 3. Tests

On a **total** machine, flipping complements the language exactly.

In [ ]:
comp = comp_dfa(even0)
from itertools import product
strs = [''.join(p) for k in range(9) for p in product('01', repeat=k)]
assert all(accepts_dfa(comp, s) == (not accepts_dfa(even0, s)) for s in strs)
print("complement verified on all %d strings up to length 8" % len(strs))
print("F before:", sorted(even0["F"]), " F after:", sorted(comp["F"]))

On a **partial** machine, flipping first gives the WRONG answer.

In [ ]:
wrong = totalize_dfa(flip(wide))     # flip, THEN totalize  <- bug
right = comp_dfa(wide)               # totalize, THEN flip  <- correct
bad = [s for s in strs if accepts_dfa(wrong, s) != (not accepts_dfa(totalize_dfa(wide), s))]
print("flip-then-totalize disagrees with the true complement on:", bad[:6])
assert bad, "the wrong order really is wrong"
print("\nshortest witness:", repr(bad[0]),
      "-- it falls into the black hole, which flip-first left NON-final")

The correct order agrees with the definition.

In [ ]:
tot = totalize_dfa(wide)
assert all(accepts_dfa(right, s) == (not accepts_dfa(tot, s)) for s in strs)
print("totalize-then-flip: correct on all %d strings" % len(strs))
print("black hole is FINAL in the complement?",
      [q for q in right["F"] if q not in wide["Q"]])

Double complement returns the original.

In [ ]:
print("comp(comp(D)) == D ?", langeq_dfa(comp_dfa(comp_dfa(even0)), even0))
assert langeq_dfa(comp_dfa(comp_dfa(even0)), even0)

## 4. Animation

The complement machine: same shape, opposite double circles.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(comp_dfa(even0), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Complement a DFA with **no** final states. What do you get?
2. Why does complementation not work this way for NFA? (Chapter 7.)
3. Write the one-line proof that $\overline{\overline{L}} = L$ for DFA.

In [ ]:
# Your work for the exercises above.